# Feature Engineering for Time Series Forecasting

The goal of this notebook is to prepare the sales time series for forecasting.

In this notebook, I will:

1. Load and check the available datasets.
2. Prepare the sales time series.
3. Clean and merge external information such as oil prices and holidays.
4. Create time-based, lag, rolling, holiday, and oil features.
5. Save the final feature dataset for modeling.

The main dataset is `timeseries.csv`, which contains daily `unit_sales`. The other datasets are used as additional context or external variables.

## Part A - Setup

I start by importing the libraries and setting the data folder.

This notebook is designed to run from the `notebooks` folder in VS Code, with the CSV files stored in the `data` folder at the project root.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Data folder:", DATA_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())

## Part B - Load the datasets

The project contains four datasets:

| Dataset | Description |
|---|---|
| `timeseries.csv` | Daily unit sales. This is the target variable. |
| `oil.csv` | Daily oil prices. This is tested as a possible external variable. |
| `holidays.csv` | Holiday information by date. |
| `stores.csv` | Store location information, used mainly for context. |

The store dataset is not merged into the main time series here because the sales dataset does not contain `store_nbr`.

In [ ]:
# Check which CSV files are available
list(DATA_DIR.glob("*.csv"))

In [ ]:
df_timeseries = pd.read_csv(DATA_DIR / "timeseries.csv", parse_dates=["date"])
df_oil = pd.read_csv(DATA_DIR / "oil.csv", parse_dates=["date"])
df_holidays = pd.read_csv(DATA_DIR / "holidays.csv", parse_dates=["date"])
df_stores = pd.read_csv(DATA_DIR / "stores.csv")

print("Data loaded successfully.")

In [ ]:
print("Time series:")
display(df_timeseries.head())

print("Oil:")
display(df_oil.head())

print("Holidays:")
display(df_holidays.head())

print("Stores:")
display(df_stores.head())

## Part C - Basic checks

Before creating features, I check the shape, data types, and missing values in each dataset.

In [ ]:
datasets = {
    "timeseries": df_timeseries,
    "oil": df_oil,
    "holidays": df_holidays,
    "stores": df_stores
}

for name, df in datasets.items():
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print(df.dtypes)
    print("Missing values:")
    print(df.isna().sum())
    print()

The basic checks help identify which datasets need cleaning before merging.

The sales dataset contains the target variable `unit_sales`. The oil dataset may have missing values because oil prices are not recorded for every calendar date. These values need to be handled before oil can be used as an external feature.

## Part D - Prepare the sales time series

For time series forecasting, the dates should be continuous and ordered.

I first sort the sales data by date and check whether any dates are missing.

In [ ]:
df_timeseries = df_timeseries.sort_values("date").reset_index(drop=True)

full_dates = pd.date_range(
    start=df_timeseries["date"].min(),
    end=df_timeseries["date"].max(),
    freq="D"
)

missing_dates = full_dates.difference(df_timeseries["date"])

print("Start date:", df_timeseries["date"].min())
print("End date:", df_timeseries["date"].max())
print("Number of missing dates:", len(missing_dates))
print(missing_dates)

If missing dates are found, they should be added to keep the time series continuous. Missing `unit_sales` values created by this step are filled with 0, assuming no sales were recorded on those days.

Even if no missing dates are found, this step ensures that the series has a strict daily frequency.

In [ ]:
df_timeseries = (
    df_timeseries
    .set_index("date")
    .asfreq("D")
)

df_timeseries["unit_sales"] = df_timeseries["unit_sales"].fillna(0)
df_timeseries = df_timeseries.reset_index()

print("Missing values after enforcing daily frequency:")
print(df_timeseries.isna().sum())

## Part E - Clean oil prices

Oil prices can be used as an external variable, but the oil series must be complete before merging it with sales.

I sort the oil data by date and fill missing oil prices using interpolation. Forward and backward filling are added to handle any remaining missing values at the start or end of the series.

In [ ]:
df_oil = df_oil.sort_values("date").reset_index(drop=True)

print("Missing oil values before cleaning:")
print(df_oil["dcoilwtico"].isna().sum())

df_oil["dcoilwtico"] = (
    df_oil["dcoilwtico"]
    .interpolate(method="linear")
    .ffill()
    .bfill()
)

print("Missing oil values after cleaning:")
print(df_oil["dcoilwtico"].isna().sum())

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_oil["date"], df_oil["dcoilwtico"])
plt.title("Oil price over time")
plt.xlabel("Date")
plt.ylabel("Oil price")
plt.show()

The oil series is now complete and can be merged with the sales data.

Oil price will be tested as a possible external feature, but its usefulness should be checked during analysis and modeling.

## Part F - Merge sales with external datasets

This is where the datasets are joined.

The sales time series is the main table. Oil prices and holiday information are added using the `date` column.

The store dataset is not merged because the current sales dataset does not contain a store identifier.

In [ ]:
# Start from the prepared sales series
df = df_timeseries.copy()

# Merge oil prices by date
df = df.merge(df_oil[["date", "dcoilwtico"]], on="date", how="left")

print("Missing values after merging oil:")
print(df[["unit_sales", "dcoilwtico"]].isna().sum())

After merging, missing oil values may appear because the sales dataset has daily dates while oil prices may not be available for every sales date.

These missing values are filled using forward filling and backward filling to keep the merged dataset complete.

In [ ]:
df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill()

print("Missing oil values after filling in merged dataset:")
print(df["dcoilwtico"].isna().sum())

display(df[["date", "unit_sales", "dcoilwtico"]].head(15))

### Add holiday information

The holiday dataset contains different holiday levels: national, regional, and local.

I create separate binary flags for each holiday type. This gives the model more information than a single holiday flag.

In [ ]:
holiday_flags = (
    df_holidays
    .assign(value=1)
    .pivot_table(
        index="date",
        columns="locale",
        values="value",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

holiday_flags.columns.name = None
holiday_flags = holiday_flags.rename(columns={
    "National": "is_national_holiday",
    "Regional": "is_regional_holiday",
    "Local": "is_local_holiday"
})

for col in ["is_national_holiday", "is_regional_holiday", "is_local_holiday"]:
    if col not in holiday_flags.columns:
        holiday_flags[col] = 0

holiday_flags = holiday_flags[["date", "is_national_holiday", "is_regional_holiday", "is_local_holiday"]]

# Merge holiday flags into the main dataset
df = df.merge(holiday_flags, on="date", how="left")

holiday_cols = ["is_national_holiday", "is_regional_holiday", "is_local_holiday"]
df[holiday_cols] = df[holiday_cols].fillna(0).astype(int)

# General holiday flag: 1 if any holiday type is present
df["is_holiday"] = (df[holiday_cols].sum(axis=1) > 0).astype(int)

print("Holiday features added.")
display(df[["date", "unit_sales", "is_holiday", "is_national_holiday", "is_regional_holiday", "is_local_holiday"]].head(15))

The merged dataset now contains sales, oil prices, and holiday indicators.

From this point onward, feature engineering can be done on one main table.

## Part G - Calendar features

Calendar features help the model capture time-based patterns, such as differences between weekdays and weekends.

In [ ]:
df["day_of_week"] = df["date"].dt.day_name()
df["day_number"] = df["date"].dt.dayofweek
df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

df["is_weekend"] = (df["day_number"] >= 5).astype(int)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

calendar_cols = [
    "date", "day_of_week", "day_number", "day", "month", "year",
    "quarter", "week_of_year", "is_weekend", "is_month_start", "is_month_end"
]

display(df[calendar_cols].head(10))

The calendar features are important because the first notebook showed a strong weekly pattern, with higher sales during weekends.

## Part H - Lag features

Lag features use previous sales values as predictors.

This is important because the ACF plot showed that past sales, especially sales from the previous week, can be useful for forecasting.

In [ ]:
df["lag_1"] = df["unit_sales"].shift(1)
df["lag_7"] = df["unit_sales"].shift(7)
df["lag_14"] = df["unit_sales"].shift(14)
df["lag_30"] = df["unit_sales"].shift(30)

lag_cols = ["date", "unit_sales", "lag_1", "lag_7", "lag_14", "lag_30"]
display(df[lag_cols].head(35))

The first rows contain missing values in the lag columns because there is not enough past data yet.

This is expected and will be handled before modeling.

## Part I - Rolling features

Rolling features summarize recent sales behavior.

To avoid data leakage, I shift sales by one day before calculating rolling statistics. This means the rolling features use only past sales, not the current day's sales.

In [ ]:
past_sales = df["unit_sales"].shift(1)

df["rolling_7d_mean"] = past_sales.rolling(window=7).mean()
df["rolling_14d_mean"] = past_sales.rolling(window=14).mean()
df["rolling_30d_mean"] = past_sales.rolling(window=30).mean()
df["rolling_7d_std"] = past_sales.rolling(window=7).std()

rolling_cols = [
    "date", "unit_sales", "rolling_7d_mean",
    "rolling_14d_mean", "rolling_30d_mean", "rolling_7d_std"
]

display(df[rolling_cols].head(35))

Rolling averages smooth short-term fluctuations and help capture recent trends in sales.

The missing values at the beginning are expected because rolling windows need enough previous observations.

## Part J - Oil features

Oil price is included as a possible external feature.

Because external effects may not happen immediately, I create lagged and rolling oil features.

In [ ]:
df["oil_lag_1"] = df["dcoilwtico"].shift(1)
df["oil_lag_7"] = df["dcoilwtico"].shift(7)
df["oil_rolling_7d_mean"] = df["dcoilwtico"].shift(1).rolling(window=7).mean()

oil_feature_cols = ["date", "dcoilwtico", "oil_lag_1", "oil_lag_7", "oil_rolling_7d_mean"]
display(df[oil_feature_cols].head(15))

In [ ]:
# Check same-day correlation between sales and oil price
corr_value = df[["unit_sales", "dcoilwtico"]].corr().iloc[0, 1]
print("Correlation between sales and oil price:", corr_value)

The oil price correlation should be interpreted carefully.

A low same-day correlation does not necessarily mean oil is useless. In time series, external effects can be delayed, so lagged oil features are tested as additional predictors.

## Part K - Check holiday effect

I compare average sales on holiday and non-holiday days as a simple first check.

In [ ]:
holiday_sales = df.groupby("is_holiday")["unit_sales"].mean()

plt.figure(figsize=(6, 4))
holiday_sales.plot(kind="bar")
plt.title("Holiday vs non-holiday sales")
plt.ylabel("Average unit sales")
plt.xticks([0, 1], ["No holiday", "Holiday"], rotation=0)
plt.show()

holiday_sales

This provides a first assessment of holiday effects.

If average sales are only slightly different between holiday and non-holiday days, the holiday effect may be limited. However, holiday flags can still be useful features for some models.

## Part L - Final feature dataset

The feature engineering process creates missing values at the beginning of the dataset because lag and rolling features require past observations.

Before modeling, these rows are removed.

In [ ]:
print("Missing values before final cleaning:")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
df_features = df.dropna().reset_index(drop=True)

print("Original rows:", len(df))
print("Rows after dropping feature-related missing values:", len(df_features))
print("Remaining missing values:", df_features.isna().sum().sum())

display(df_features.head())

Rows with missing feature values were removed only after all lag and rolling features were created.

This keeps the feature dataset clean and ready for modeling.

## Part M - Save the final dataset

The final feature dataset is saved as a CSV file so it can be used in the modeling notebook.

In [ ]:
output_path = DATA_DIR / "timeseries_with_features.csv"
df_features.to_csv(output_path, index=False)

print("Feature dataset saved to:", output_path.resolve())
print("Final shape:", df_features.shape)

## Summary

Main points from this notebook:

- The sales series was prepared as a continuous daily time series.
- Oil prices were cleaned and merged with the sales data.
- Holiday indicators were added using the holiday dataset.
- Calendar features were created to capture weekday, weekend, month, and year effects.
- Lag features were created to include past sales information.
- Rolling features were created to capture recent sales behavior.
- Oil lag and rolling features were added as possible external predictors.
- Rows with missing values created by lag and rolling calculations were removed before saving the final dataset.

The final dataset is ready to be used for modeling.